# APO Benchmark Analysis

Interactive analysis of optimizer benchmark results.

In [ ]:
import json
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Load results
with open('../benchmark_results/results.json', 'r') as f:
    data = json.load(f)

results = data['results']
print(f"Loaded {len(results)} benchmark results")
print(f"Timestamp: {data['timestamp']}")

## Summary Statistics

In [ ]:
df = pd.DataFrame(results)
summary = df.groupby('optimizer_name').agg({
    'improvement': ['mean', 'std', 'min', 'max'],
    'final_score': ['mean', 'std'],
    'duration_seconds': ['mean', 'std'],
    'total_evaluations': ['mean', 'std']
}).round(4)

summary

## Performance by Task

In [ ]:
pivot = df.pivot_table(
    values='final_score',
    index='optimizer_name',
    columns='task_name',
    aggfunc='mean'
)

pivot.style.background_gradient(cmap='RdYlGn', axis=None)

## Convergence Analysis

In [ ]:
# Find fastest converging optimizers
convergence_df = df[df['convergence_iteration'].notna()][[
    'optimizer_name', 'task_name', 'convergence_iteration', 'final_score'
]].sort_values('convergence_iteration')

print("Fastest Convergence (fewest iterations to best score):")
convergence_df.head(10)

## Efficiency Rankings

In [ ]:
# Calculate efficiency score: improvement / (time * evaluations)
df['efficiency'] = df['improvement'] / (df['duration_seconds'] * df['total_evaluations'] + 1)

efficiency_rankings = df.groupby('optimizer_name')['efficiency'].mean().sort_values(ascending=False)

fig = go.Figure(go.Bar(
    x=efficiency_rankings.index,
    y=efficiency_rankings.values,
    marker_color='lightblue'
))

fig.update_layout(
    title="Optimizer Efficiency Score",
    xaxis_title="Optimizer",
    yaxis_title="Efficiency (improvement/cost)",
    height=400
)

fig.show()

## Best Prompts

In [ ]:
# Show best prompts for each task
for task in df['task_name'].unique():
    task_df = df[df['task_name'] == task]
    best = task_df.loc[task_df['final_score'].idxmax()]
    
    print(f"\n{'='*60}")
    print(f"Task: {task}")
    print(f"Best Optimizer: {best['optimizer_name']}")
    print(f"Score: {best['initial_score']:.1%} → {best['final_score']:.1%}")
    print(f"\nBest Prompt:")
    print(best['best_prompt'])
    print(f"{'='*60}")

## Statistical Significance

In [ ]:
from scipy import stats

# Compare top 2 optimizers
top_optimizers = df.groupby('optimizer_name')['final_score'].mean().nlargest(2)
opt1, opt2 = top_optimizers.index[0], top_optimizers.index[1]

scores1 = df[df['optimizer_name'] == opt1]['final_score']
scores2 = df[df['optimizer_name'] == opt2]['final_score']

t_stat, p_value = stats.ttest_ind(scores1, scores2)

print(f"Comparing {opt1} vs {opt2}:")
print(f"Mean scores: {scores1.mean():.3f} vs {scores2.mean():.3f}")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")
print(f"Significant? {'Yes' if p_value < 0.05 else 'No'} (α=0.05)")

## Recommendations

Based on benchmark results, choose optimizer based on your needs:

In [ ]:
# Generate recommendations
recommendations = {
    "Best Overall Performance": df.groupby('optimizer_name')['final_score'].mean().idxmax(),
    "Fastest (least time)": df.groupby('optimizer_name')['duration_seconds'].mean().idxmin(),
    "Most Efficient": df.groupby('optimizer_name')['efficiency'].mean().idxmax(),
    "Most Improved": df.groupby('optimizer_name')['improvement'].mean().idxmax(),
    "Most Sample Efficient": df.groupby('optimizer_name')['total_evaluations'].mean().idxmin(),
}

for category, optimizer in recommendations.items():
    print(f"{category}: {optimizer}")